In [1]:
import sys, platform, importlib.metadata, json
print("AUDIT_KERNEL="+json.dumps({"python":sys.version,"executable":sys.executable,"platform":platform.platform(),"packages":dict(sorted((d.metadata["Name"],d.version) for d in importlib.metadata.distributions()))}))

AUDIT_KERNEL={"python": "3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]", "executable": "C:\\Users\\Miakhlov\\Documents\\GitHub\\Computational chemistry_course\\tmp\\course-restructure-20261118\\scripts\\environment\\.venv-chem\\Scripts\\python.exe", "platform": "Windows-10-10.0.19045-SP0", "packages": {"Jinja2": "3.1.6", "MarkupSafe": "3.0.4", "PyYAML": "6.0.3", "Pygments": "2.21.0", "Send2Trash": "2.2.0", "altair": "6.3.0", "anyio": "4.16.0", "argon2-cffi": "25.1.0", "argon2-cffi-bindings": "26.2.0", "arrow": "1.4.0", "asttokens": "3.0.2", "async-lru": "2.4.0", "attrs": "26.1.0", "babel": "2.18.0", "beautifulsoup4": "4.15.0", "bleach": "6.4.0", "certifi": "2026.10.23", "cffi": "2.1.1", "charset-normalizer": "3.5.2", "click": "8.5.1", "cloudpickle": "3.1.3", "colorama": "0.4.6", "comm": "0.2.3", "contourpy": "1.4.0", "cycler": "0.12.1", "debugpy": "1.8.22", "defusedxml": "0.7.1", "executing": "2.3.0", "fastjsonschema": "2.22.3", "fonttools": "4.68.0

# Разбор A. От энергии к физическому выводу

У нас уже есть сохранённые PySCF-вычисления трёх молекул. Прочитать число из файла проще, чем проверить, относится ли оно к нужной геометрии. Гипотеза: явная связь geometry_sha256, метода и сходимости позволяет обнаружить подмену входа. На линии A проверяем истрадефиллин; B независимо проверяет аторвастатин. Этот урок анализирует фактический архив, а не заявляет повторный запуск PySCF на Windows.

Прочитайте result.json и исходную geometry.json, пересчитайте SHA-256. Проверьте SCF, charge, spin, метод и число переходов. Сопоставьте файл с логом и версиями. Полный Linux-маршрут: установите quantum и выполните scripts/calculate_quantum_reference.py --help; запускайте его в новом каталоге, не заменяя reference.

In [2]:
from pathlib import Path
import sys,json,math
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'programm/course.json').is_file())
sys.path.insert(0,str(ROOT/'practicuum/shared'))
from course_support import DATA,compound,esol,workspace,write_json,init_dossier
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
def publish_plot(fig,name):
    fig.savefig(OUT/name,dpi=120,bbox_inches='tight')
    display(fig)
    plt.close(fig)
OUT=workspace('M07-L02','case')


Сначала проверяем входы и единицы. Словари сохраняют названия полей вместе со значениями; список сохраняет соответствие записей. Выведите промежуточный результат перед дальнейшим анализом.

In [3]:
import hashlib
qfolder=DATA/'quantum'/'istradefylline'
q=json.loads((qfolder/'result.json').read_text(encoding='utf-8'))
actual=hashlib.sha256((qfolder/'geometry.json').read_bytes()).hexdigest()
assert actual==q['geometry_sha256'] and q['scf_converged']
print({k:q[k] for k in ['compound','method','charge','spin','energy_hartree','geometry_method']})


{'compound': 'istradefylline', 'method': 'B3LYP/STO-3G, density fitting, grids level 1, gas phase; TDDFT singlets, length gauge', 'charge': 0, 'spin': 0, 'energy_hartree': -1279.956558216535, 'geometry_method': 'ETKDGv3 seed 20261118 + MMFF; not DFT optimized'}


Теперь сравним модель с контрольным случаем. `assert` останавливает расчёт при нарушении контракта; это защита от ошибочного продолжения, а не оценка научного понимания.

In [4]:
assert len(q['transitions'])==6 and q['charge']==0 and q['spin']==0
for t in q['transitions']:
    assert math.isclose(t['wavelength_nm']*t['energy_eV'],1239.8419843320026,rel_tol=1e-6)
display(q['transitions']);display(q['limitations'])


[{'state': 1,
  'energy_eV': 3.7122936942342766,
  'wavelength_nm': 333.98273047675474,
  'oscillator_strength': 0.002695716220243701},
 {'state': 2,
  'energy_eV': 3.81325282383995,
  'wavelength_nm': 325.1402520652906,
  'oscillator_strength': 0.592037869830763},
 {'state': 3,
  'energy_eV': 4.077882284243059,
  'wavelength_nm': 304.04065098268114,
  'oscillator_strength': 0.07553527060244697},
 {'state': 4,
  'energy_eV': 4.549126489033398,
  'wavelength_nm': 272.54506712901826,
  'oscillator_strength': 0.005276339172888101},
 {'state': 5,
  'energy_eV': 4.872509146198535,
  'wavelength_nm': 254.45657404241308,
  'oscillator_strength': 0.0019162899667283947},
 {'state': 6,
  'energy_eV': 4.875753842518963,
  'wavelength_nm': 254.28723934337557,
  'oscillator_strength': 0.2471052047431396}]

['minimal basis and coarse grid',
 'no solvent',
 'no vibrational/thermal broadening',
 'MMFF geometry, not DFT stationary point',
 'no absolute measured extinction coefficient',
 'not a reproduction of the literature aqueous-xanthine protocol']

Сохраним именно полученный результат. Не переносите этот вывод на другую химическую форму, мишень или экспериментальные условия без отдельной проверки.

In [5]:
result={'compound':q['compound'],'geometry_sha256':actual,'archive_scf_converged':q['scf_converged'],
        'kind':'calculated','PySCF_rerun_this_lesson':'NOT_RUN','method':q['method']}


In [6]:
write_json(OUT/'result.json',result)
from dossier_records import append_record
receipt=append_record(init_dossier(),'M07-L02','A',result,ROOT,inputs=['practicuum/shared/data/quantum/istradefylline/result.json', 'practicuum/shared/data/quantum/istradefylline/geometry.json', 'practicuum/shared/data/quantum/atorvastatin/result.json', 'practicuum/shared/data/quantum/atorvastatin/geometry.json'])
print('Versioned evidence:',receipt)


Versioned evidence: C:\Users\Miakhlov\Documents\GitHub\Computational chemistry_course\tmp\course-restructure-20261118\practicuum\evidence\20261118-full-course-windows\workspace\my_research\evidence\M07-L02\20261118T141606192769Z-A-39c140d17a8c4e6e8b9d3b74fae9b06d.json


Хеш связывает байты геометрии с результатом; он не доказывает правильный выбор конформера. Открытый вопрос — насколько спектральный отклик изменится при растворителе и базисе. M08 превращает переходы в явно условный спектр.

Перед переходом ответьте: какие результаты получены вычислением, какие заданы условно, и какая независимая проверка ещё отсутствует?